# Frisbee detection: YOLO26s + live tracking

End-to-end starter for a **fixed field camera + desktop GPU**. We will:

1. Download [DG Putting v1](https://universe.roboflow.com/dustin-klein/dg-putting/dataset/1) from Roboflow.
2. Inspect dataset classes and images.
3. Fine-tune `yolo26s.pt` on the original five labels.
4. Evaluate the model on validation/test images.
5. Detect **`disc_flying`** and **`disc_ground`** in an image, video, or webcam stream.

Run this notebook from the **same folder** as `frisbee_pipeline.py`. Download/training requires internet and (for the dataset) a Roboflow account API key; no credentials are saved in the notebook.

## 0. Installation

Run the next cell in a new environment. If you use an NVIDIA GPU, install a CUDA-enabled PyTorch build appropriate for your machine first: [PyTorch installation guide](https://pytorch.org/get-started/locally/). Restart the notebook kernel after installing packages if prompted.

In [ ]:
%pip install -r requirements.txt

  Using cached matplotlib-3.11.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached contourpy-1.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 205.7 kB/s  0:00:06185.3 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 1.8 MB/s  0:04:13 eta 0:00:010:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 3.9 MB/s  0:03:00 eta 0:00:010:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 2.1 MB/s  0:01:07 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 2.5 MB/s  0:01:32 eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 2.6 MB/s

In [ ]:
from pathlib import Path
from getpass import getpass
import os

import torch
import yaml
from PIL import Image
from IPython.display import display

from frisbee_pipeline import (
    download_dataset, train_model, evaluate_model,
    predict_image, run_video, disc_class_ids,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA not found. Training will be slower on CPU.")

## 1. Download DG Putting v1

The SDK needs a **Roboflow API key**. Set `ROBOFLOW_API_KEY` in your environment, or enter it via the hidden prompt below. If the dataset is already present in `data/dg-putting-v1`, no API key is needed. We create a local YAML file with correct absolute data paths.

In [ ]:
DATA_DIR = Path("data/dg-putting-v1")
api_key = os.environ.get("ROBOFLOW_API_KEY")
if not (DATA_DIR / "data.yaml").exists() and not api_key:
    api_key = getpass("Roboflow API key (input hidden): ")

DATA_YAML = download_dataset(DATA_DIR, api_key=api_key)
print("Local YOLO dataset configuration:", DATA_YAML)
config = yaml.safe_load(DATA_YAML.read_text())
print("Classes:", config["names"])
print("Train:", config["train"], "| Validation:", config["val"],
      "| Test:", config.get("test", "not available"))

## 2. Inspect some dataset images

View a few original images before training. The classes are `basket`, `person`, `disc_flying`, `disc_ground`, and `disc_in_basket` (check your own `config["names"]` for the exact order). The exact class indices are read from the YAML/model, not hard-coded.

In [ ]:
root = Path(config["path"])
train_images = sorted(
    p for p in (root / config["train"]).iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print("Training images:", len(train_images))
for path in train_images[:3]:
    print(path.name)
    im = Image.open(path)
    im.thumbnail((550, 350))
    display(im)

## 3. Fine-tune YOLO26s

Defaults target a desktop GPU: `imgsz=960`, `batch=4`, `epochs=50`. Start with a one-epoch smoke test (`EPOCHS=1`) if you want to verify the setup. Reduce `BATCH` or `IMGSZ` if you run out of VRAM. **The first run downloads pretrained weights.**

In [ ]:
MODEL = "yolo26s.pt"   # alternatives: yolo26n.pt, yolo26m.pt
EPOCHS = 50
IMGSZ = 960
BATCH = 4
RUN_NAME = "yolo26s_960"

best_weights = train_model(
    data_yaml=DATA_YAML,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    run_name=RUN_NAME,
)
print("Best weights:", best_weights)

## 4. Evaluation

For frequent hyperparameter changes use `split="val"`. Evaluate `split="test"` on the held-out test set only after choosing your final configuration. Below we use validation by default.

In [ ]:
metrics = evaluate_model(
    weights=best_weights,
    data_yaml=DATA_YAML,
    imgsz=IMGSZ,
    split="val",  # change to "test" for final held-out evaluation
)
metrics

## 5. Inference on an image

The inference helper dynamically selects `disc_flying` and `disc_ground` by **name**. It writes an annotated image under `results/`. Try one of the held-out dataset images or replace `sample_path` with your own image.

In [ ]:
test_dir = root / config.get("test", config["val"])
images = sorted(p for p in test_dir.iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert images, f"No images in {test_dir}"
sample_path = images[0]
output_path, detections = predict_image(
    weights=best_weights,
    image_path=sample_path,
    save_path="results/sample_prediction.jpg",
    imgsz=IMGSZ,
    conf=0.15,
)
print("Detected:", detections)
display(Image.open(output_path))

## 6. Video or webcam inference + ByteTrack

The following calls are **intentionally commented out** so executing all notebook cells won't open a live window or block the kernel. Uncomment one on your local machine. Press **Q** or **ESC** to exit the live window. On headless systems, use `show=False` with `save_path`.

- `source=0` means the default webcam; try `source=1` for an external camera.
- Use `source="your_video.mp4"` for recorded footage.
- `tracker="ocsort.yaml"` is an alternative for fast, non-linear motion.
- Track IDs can change when `disc_flying` becomes `disc_ground`.

In [ ]:
# Webcam tracking (uncomment to run locally):
# run_video(best_weights, source=0, imgsz=IMGSZ, conf=0.10, show=True)

# Video file: save annotated output without opening an OpenCV window:
# run_video(best_weights, source="my_throw.mp4", imgsz=IMGSZ,
#           conf=0.10, show=False, save_path="results/tracked_throw.mp4")

## 7. Next steps for a fixed field camera

- Capture frames from your actual field camera, including **tiny flying discs, edge-on discs, motion blur, landings, grass, and empty frames**.
- Split by *throw or recording session*, not individual neighboring frames.
- Compare `IMGSZ=640`, `960`, and `1280`; measure detection recall by disc size and true end-to-end FPS.
- For very small distant discs, consider tiled inference or field/sky crops; these increase computation and should be benchmarked.
- Tune `conf`, compare ByteTrack vs OC-SORT, and add temporal state smoothing if tracking a landing event is important.

Sources: [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26), [Ultralytics tracking](https://docs.ultralytics.com/modes/track), [DG Putting v1 dataset](https://universe.roboflow.com/dustin-klein/dg-putting/dataset/1) (**CC BY 4.0**).